# LEAF: second accelerator (Kaggle T4)

Guide: `scripts/kaggle/README.md`. Settings: **Accelerator = GPU T4 x2** (timing uses card 0 only; card 1 stays idle), **Internet = on**, secret **GITHUB_TOKEN** attached. Run with **Save Version → Save & Run All (Commit)**; the archive appears under *Output*.

- `PHASE = 'device-1'`: smoke test + timing of the workload set → `second_device_1.tar.gz`
- `PHASE = 'device-2'`: timing of every planned arm → `second_device_3.tar.gz` (needs the dataset with `second_device_2.tar.gz`)

In [ ]:
PHASE = 'device-1'                                   # 'device-1' (session 1) or 'device-2' (session 2)
ARCHIVE = '/kaggle/input/leaf-second-device/second_device_2.bin'   # device-2 only: upload second_device_2.tar.gz renamed to .bin
BRANCH = 'dev'

In [ ]:
!nvidia-smi
!nvidia-smi --query-compute-apps=pid,gpu_bus_id --format=csv
import torch; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
assert 'T4' in torch.cuda.get_device_name(0), 'choose a T4 accelerator'
# Kaggle offers only 'GPU T4 x2': timing runs on card 0 (CUDA_VISIBLE_DEVICES=0, set by the script); card 1 stays idle.
print(torch.cuda.device_count(), 'cards; timing on card 0, the others are left idle')

In [ ]:
import os, glob, subprocess, shutil
subprocess.run(['rm', '-rf', '/tmp/LEAF'], check=True)
# Source code: an attached dataset with LEAF_code.zip (no token needed), otherwise a clone with the GITHUB_TOKEN secret.
zips = glob.glob('/kaggle/input/**/LEAF_code.zip', recursive=True)
dirs = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/leaf/reproduction/experiments/second_device.py', recursive=True)]
if zips:
    subprocess.run(['unzip', '-q', zips[0], '-d', '/tmp/LEAF'], check=True); print('code from', zips[0])
elif dirs:
    shutil.copytree(os.path.dirname(os.path.dirname(os.path.dirname(dirs[0]))), '/tmp/LEAF'); print('code from', dirs[0])
else:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    r = subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', BRANCH, f'https://{token}@github.com/huynhcanhvien/LEAF.git', '/tmp/LEAF'], capture_output=True, text=True)
    print('clone ok' if r.returncode == 0 else r.stderr.replace(token, '***')); del token
os.chdir('/tmp/LEAF'); print(open('KAGGLE_CODE_VERSION').read() if os.path.exists('KAGGLE_CODE_VERSION') else subprocess.check_output(['git', 'log', '--oneline', '-1'], text=True))

In [ ]:
!pip install -q 'timm==1.0.30' scipy python-dotenv
!pip install -q -e . --no-deps

In [ ]:
import os, glob, subprocess
if PHASE == 'device-2' and not os.path.exists(ARCHIVE):
    # Kaggle extracts an uploaded .tar.gz (and refuses its '#' file names): upload it renamed to .bin; find it anywhere
    found = [f for f in glob.glob('/kaggle/input/**/second_device_2*', recursive=True) if f.endswith(('.bin', '.tar.gz'))]
    roots = [p for p in glob.glob('/kaggle/input/**/results/second_device', recursive=True)]
    if found: ARCHIVE = found[0]
    elif roots:
        base = os.path.dirname(os.path.dirname(roots[0])); ARCHIVE = '/tmp/second_device_2.tar.gz'
        subprocess.run(['tar', 'czf', ARCHIVE, '-C', base, 'results/second_device'], check=True)
    else: raise SystemExit('attach the dataset holding second_device_2.tar.gz')
    print('archive:', ARCHIVE)
import os
cmd = ['bash', 'scripts/run_second_device.sh', PHASE] + ([ARCHIVE] if PHASE == 'device-2' else [])
# Progress is printed live (shown in the log of a running version) and kept in /kaggle/working/<PHASE>.log.
with open(f'/kaggle/working/{PHASE}.log', 'w') as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        log.write(line); log.flush()
        if 'Warning' not in line: print(line, end='', flush=True)
    rc = proc.wait()
assert rc == 0, f'{PHASE} failed (exit {rc}); see /kaggle/working/{PHASE}.log'

In [ ]:
import glob, shutil
for f in glob.glob('/tmp/LEAF/second_device_*.tar.gz'): shutil.copy(f, '/kaggle/working/'); print('output:', f)
shutil.copy('/tmp/LEAF/results/second_device/device_check.json', '/kaggle/working/') if PHASE == 'device-1' else None